In [266]:
import sys
!{sys.executable} -m pip install xgboost


[notice] A new release of pip is available: 25.0 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [267]:
import re
import string
import xgboost as xgb
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [268]:
df_true = pd.read_csv('/Users/hwey/Desktop/projects/faux-lens/notebooks/data/True.csv')
df_fake = pd.read_csv('/Users/hwey/Desktop/projects/faux-lens/notebooks/data/Fake.csv')

In [269]:
def processText(text):
    text = re.sub(r'^.*?\(.*?\)\s*-\s*', '', text)
    artifacts = [
        'reuters', 'getty', 'images', 'image', 'pic', 'photo', 'credit', 
        'watch', 'video', 'video_player', 'read', 'click', 'link', 
        'via', 'featured', 'filessupport', 'contributed', 'reporting'
    ]
    pattern = r'\b(' + '|'.join(artifacts) + r')\b'
    text = re.sub(pattern, '', text, flags = re.IGNORECASE)
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\b\S+\.(com|net|org|edu|gov)\b', '', text)
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'\[.*?\]', ' ', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), ' ', text)
    text = re.sub(r'\W', " ", text)
    text = re.sub(r'\w*\d\w*', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [270]:
df_true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [271]:
df_fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [272]:
print(f'True news dataset size: {df_true.shape} and False news dataset size: {df_fake.shape}')

True news dataset size: (21417, 4) and False news dataset size: (23481, 4)


In [273]:
print(f"Number of unique subjects for real news: {df_true["subject"].nunique()}")
print(f"Number of unique subjects for fake news: {df_fake["subject"].nunique()}")

Number of unique subjects for real news: 2
Number of unique subjects for fake news: 6


In [274]:
df_fake["subject"].value_counts()

subject
News               9050
politics           6841
left-news          4459
Government News    1570
US_News             783
Middle-east         778
Name: count, dtype: int64

In [275]:
df_true["subject"].value_counts() 

subject
politicsNews    11272
worldnews       10145
Name: count, dtype: int64

In [276]:
df_true = df_true.drop("subject", axis=1)
df_fake = df_fake.drop("subject", axis=1)

In [277]:
df_true["class"] = 1
df_fake["class"] = 0

In [278]:
df = pd.concat([df_true, df_fake], axis=0, ignore_index=True)
df.head()

,title,text,date,class
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,"December 29, 2017",1


In [279]:
df["full_content"] = df["title"].str.cat(df["text"], sep=" ", na_rep = '')
df["full_content"] = df["full_content"].apply(processText)

In [280]:
df = df.drop(["title", "text", "date"], axis=1)
df.head()

,class,full_content
0,1,the head of a conservative republican faction ...
1,1,transgender people will be allowed for the fir...
2,1,the special counsel investigation of links bet...
3,1,trump campaign adviser george papadopoulos tol...
4,1,president donald trump called on the u s posta...


In [281]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(
    df, 
    test_size=0.2,
    shuffle=True,
    stratify=df['class'],
    random_state=42
)

In [282]:
print("New y_train unique values:", train_df['class'].unique())
print("New y_test unique values:", test_df['class'].unique())

New y_train unique values: [1 0]
New y_test unique values: [1 0]


In [283]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')
X_train = tfidf.fit_transform(train_df['full_content'])
X_test = tfidf.transform(test_df['full_content'])
y_train = train_df['class'].values
y_test = test_df['class'].values

In [ ]:
from xgboost import XGBClassifier
model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    objective='binary:logistic',
    random_state=42,
    eval_metric='logloss'
)
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [285]:
y_pred = model.predict(X_test)

In [286]:
from sklearn.metrics import classification_report
report = classification_report(y_test, y_pred, target_names=['Fake', 'True'])
print("--- XGBoost Classification Report ---")
print(report)

--- XGBoost Classification Report ---
              precision    recall  f1-score   support

        Fake       0.98      0.98      0.98      4696
        True       0.98      0.98      0.98      4284

    accuracy                           0.98      8980
   macro avg       0.98      0.98      0.98      8980
weighted avg       0.98      0.98      0.98      8980



In [287]:
import joblib
model.save_model("model_v1.json")
print("---SAVED MODEL AS model_v1.json---")
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
print("---SAVED DATA VECTORIZER AS tfidf_vectorizer.pkl---")

---SAVED MODEL AS model_v1.json---
---SAVED DATA VECTORIZER AS tfidf_vectorizer.pkl---
